# q client for DB Service
The DB Service q client provides a thin wrapper over the DB Service APIs, making it easier to connect to a running service and perform client operations from q.

Use this client to create a session, run queries, manage tables, and interact with DB Service from q.

## Requirements
- kdb-x
- A running DB Service instance

## Initialize Runtime

In [1]:
# Load PyKX for q notebook cells
import pykx as kx

## Load the client
From the `dbservice-qclient` repository root, load the client:

In [2]:
%%q
dbs:use`kx.dbservice_client
session:dbs.createSession[]

## Endpoint Configuration
The client uses `http://localhost:8080` as the default base URL.

Create a session with the default base URL: `session:dbs.createSession[]`

Or set the base URL explicitly: `session:dbs.createSession["localhost:8080"]`

## Managing Tables
Use these calls to define and inspect table schemas in DB Service.

In [3]:
%%q
// List tables (empty to begin with)
session.listTables[]

()


### Reference data and foreign keys

`instruments` is a reference table: a small, slowly changing table of instrument metadata, keyed on `sym` using `primaryKeys`. List the key column first in `columns` so that the schema matches the column order of the keyed table.

Declaring `foreign` as `instruments.sym` on the `fxquote.sym` column links the quotes to that reference data, which lets a query read `instruments` columns using dot notation, for example `instruments.category`.

In [4]:
%%q
// Define columns
instrumentsCols:(`name`type!("sym";"symbol");`name`type!("instrumentid";"long");`name`type!("category";"symbol");`name`type!("decimals";"long");`name`type!("pipdecimals";"long"))

// Create the 'instruments' reference table, keyed on 'sym'
session.createTable[`table`type`primaryKeys`columns!("instruments";"splayed";enlist "sym";instrumentsCols)]

jobId     | "512ec101-f1bc-b9bd-08fe-dee164c613ef"
status    | "completed"
statusUri | "/api/v0/jobs/512ec101-f1bc-b9bd-08fe-dee164c613ef"
startedAt | "2026-09-02T15:42:18.205845159"
finishedAt| 0n
table     | ()
warnings  | ()


In [5]:
%%q
// Define columns
fxquoteCols:(`name`type!("trddate";"date");`name`type!("ts";"timestamp");`name`type`foreign`attrMem`attrDisk`attrOrd!("sym";"symbol";"instruments.sym";"grouped";"parted";"parted");`name`type!("bid";"float");`name`type!("ask";"float"))

// Create partitioned table ('fxquote'), with 'sym' as a foreign key into 'instruments'
session.createTable[`table`type`prtnCol`sortColsDisk`sortColsOrd`columns!("fxquote";"partitioned";"ts";enlist "sym";enlist "sym";fxquoteCols)]

jobId     | "6ed8545d-b5d5-6e0b-0182-c4c40221d6fa"
status    | "completed"
statusUri | "/api/v0/jobs/6ed8545d-b5d5-6e0b-0182-c4c40221d6fa"
startedAt | "2026-09-02T15:42:27.283523512"
finishedAt| 0n
table     | ()
warnings  | ()


In [6]:
%%q
// List tables ('fxquote' and 'instruments' tables returned)
session.listTables[]

"instruments"
"fxquote"


In [7]:
%%q
// Describe the 'fxquote' table
session.describeTable["fxquote"]

type        | "partitioned"
prtnCol     | "ts"
sortColsDisk| ,"sym"
sortColsOrd | ,"sym"
columns     | (`name`type!("trddate";"date");`name`type!("ts";"timestamp");`n..
name        | "fxquote"


## Importing Data
DB Service supports both `file-based` and `in-memory` ingest.
Any file you want to import must first be copied into the DB Service `imports` staging directory, for example: `~/.kx/db-service/data/imports/`

### Import CSV

In [8]:
%%q
// Import a CSV file into the existing 'fxquote' table
job:session.importFiles[`table`path!("fxquote";"fxquote.csv.gz")]

In [9]:
%%q
// Check the status of the above import job
session.getImport[job`jobId]

jobId              | "8dda01ce-6113-5488-57d0-a99e82ea3870"
database           | "db"
jobType            | "import"
status             | "completed"
affectedTables     | ,"fxquote"
processedPartitions| ,"2026-03-02"
progress           | `currentPartition`partitionIndex`partitionTotal`currentT..
error              | ""
warnings           | ()
updated            | "2026-09-02T15:42:45.338904617"


In [10]:
%%q 
// Import a parquet file into the existing 'fxquote' table
job:session.importFiles[`table`path!("fxquote";"fxquote.parquet")]

In [11]:
%%q
// Import a CSV file into the 'instruments' reference table created above.
// (createTable would create a missing table from the file, but a reference
//  table needs the primary key that only an explicit schema can declare.)
job:session.importFiles[`table`path!("instruments";"instruments.csv")]

### Import JSON
Import rows directly from q without file staging.

In [12]:
%%q
// Objects payload imported to 'instruments' table
data:((`instrumentid`sym`category`decimals`pipdecimals)!(77;"USDBRL";"EM";4;4);(`instrumentid`sym`category`decimals`pipdecimals)!(78;"USDKRW";"EM";2;2));
job:session.importData[`table`data!(`instruments;data)]

In [13]:
%%q
// Rows payload imported to the 'fxquote' table
data:(("2026-01-21";"2026-01-21T10:00:00.000";"EURUSD";901.2;901.3);("2026-01-21";"2026-01-21T10:00:00.000";"EURUSD";901.2;901.3));
job:session.importData[`table`data`columnNames!(`fxquote;data;(`trddate`ts`sym`bid`ask))]

// Note: for rows payload, columnNames are required.

## Querying Tables
Run structured, SQL, or q queries against DB Service.

In [14]:
%%q
// Structured query
session.querySimple[`table`startTS`endTS`sortCols`limit!(`fxquote;2026.03.02D;2026.03.03D;enlist "ts"; 5)]

trddate    ts                            sym    bid     ask    
---------------------------------------------------------------
2026.03.02 2026.03.02D00:00:00.000000000 AUDUSD 0.67091 0.67094
2026.03.02 2026.03.02D00:00:00.000000000 EURUSD 1.16397 1.16399
2026.03.02 2026.03.02D00:00:00.000000000 GBPUSD 1.3419  1.34194
2026.03.02 2026.03.02D00:00:00.000000000 USDCAD 1.38744 1.3875 
2026.03.02 2026.03.02D00:00:00.000000000 USDJPY 158.162 158.167


A foreign key lets a structured query reach into reference data using `table.column` dot notation. Dot columns can be used in `agg`, `groupBy` and `filter`, and are returned under their dotted name.

In [15]:
%%q
// Structured query joining reference data over the 'sym' foreign key:
// 'instruments.category' is returned alongside the quotes, and is filtered on 'Major'
session.querySimple[`table`startTS`endTS`agg`filter`sortCols!(`fxquote;2026.03.02D00:00:00;2026.03.02D00:00:10;("ts";"sym";"bid";"ask";"instruments.category");enlist("=";"instruments.category";"Major");enlist "ts")]

ts                            sym    bid     ask     instruments.category
-------------------------------------------------------------------------
2026.03.02D00:00:00.000000000 EURUSD 1.16397 1.16399 Major               
2026.03.02D00:00:00.000000000 GBPUSD 1.3419  1.34194 Major               
2026.03.02D00:00:00.000000000 USDCAD 1.38744 1.3875  Major               
2026.03.02D00:00:00.000000000 USDJPY 158.162 158.167 Major               
2026.03.02D00:00:01.000000000 GBPUSD 1.34189 1.34194 Major               
2026.03.02D00:00:01.000000000 USDJPY 158.163 158.17  Major               
2026.03.02D00:00:02.000000000 USDJPY 158.166 158.17  Major               
2026.03.02D00:00:03.000000000 GBPUSD 1.34187 1.34191 Major               
2026.03.02D00:00:06.000000000 USDCAD 1.38741 1.38746 Major               
2026.03.02D00:00:06.000000000 USDJPY 158.163 158.165 Major               
2026.03.02D00:00:07.000000000 EURUSD 1.16398 1.16401 Major               
2026.03.02D00:00:07.000000000 USDJPY 1

In [16]:
%%q
// SQL query
session.querySQL[enlist[`query]!enlist "SELECT * FROM instruments WHERE category LIKE 'EM'"]

sym    instrumentid category decimals pipdecimals
-------------------------------------------------
CHFZAR 14           EM       5        4          
EURTRY 29           EM       5        4          
EURZAR 31           EM       5        4          
GBPZAR 41           EM       5        4          
USDCNH 61           EM       5        4          
USDINR 66           EM       5        4          
USDMXN 68           EM       5        4          
USDTHB 73           EM       3        2          
USDTRY 74           EM       5        4          
USDZAR 75           EM       5        4          
ZARJPY 76           EM       3        2          
USDBRL 77           EM       4        4          
USDKRW 78           EM       2        2          


In [17]:
%%q
// QSQL query
session.queryQ[enlist[`query]!enlist "select o:first bid,h:max bid,l:min bid,c:last bid by trddate,sym from fxquote"]

trddate    sym   | o       h       l       c      
-----------------| -------------------------------
2026.01.21 EURUSD| 901.2   901.2   901.2   901.2  
2026.03.02 AUDUSD| 0.67091 0.67469 0.67065 0.67307
2026.03.02 EURUSD| 1.16397 1.1768  1.16325 1.17268
2026.03.02 GBPUSD| 1.3419  1.34913 1.34101 1.34404
2026.03.02 USDCAD| 1.38744 1.3879  1.3814  1.38336
2026.03.02 USDJPY| 158.162 158.602 157.476 158.151
2026.03.03 AUDUSD| 0.6731  0.67781 0.67273 0.6754 
2026.03.03 EURUSD| 1.17258 1.1743  1.16703 1.16726
2026.03.03 GBPUSD| 1.34401 1.34588 1.33996 1.34176
2026.03.03 USDCAD| 1.38338 1.38441 1.37855 1.3844 
2026.03.03 USDJPY| 158.177 158.529 157.746 158.461


## Deleting tables
A table that is the target of a foreign key cannot be dropped while the referencing table still exists, so drop `fxquote` before `instruments`.

In [18]:
%%q
// List tables (expected: 'fxquote' and 'instruments')
session.listTables[]

"instruments"
"fxquote"


In [19]:
%%q
// Drop the 'fxquote' table (the table holding the foreign key)
session.dropTable["fxquote"]

jobId     | "a4fcf4d3-6dd1-118d-7385-89e3755fc4b2"
status    | "completed"
statusUri | "/api/v0/jobs/a4fcf4d3-6dd1-118d-7385-89e3755fc4b2"
startedAt | "2026-09-02T15:44:51.408583376"
finishedAt| 0n
table     | ()
warnings  | ()


In [20]:
%%q
// Drop the 'instruments' table
session.dropTable["instruments"]

jobId     | "d014aaa5-d876-82ff-3739-c549d40befa1"
status    | "completed"
statusUri | "/api/v0/jobs/d014aaa5-d876-82ff-3739-c549d40befa1"
startedAt | "2026-09-02T15:45:00.352968061"
finishedAt| 0n
table     | ()
warnings  | ()


In [21]:
%%q
// List tables (expected: both tables are gone)
session.listTables[]

()
